# Part A: Build the Bookstore Backend

**Aim:** Build a small, reliable database and Python functions that a language model will call in later lab parts. **No LLM is needed yet.**

**Learning outcomes.** By the end of Part A you should be able to (1) explain the three database tables and their relationships, (2) inspect catalog data in an interactive Colab table, (3) call each API function directly, and (4) distinguish reading data from changing state.

**Execution convention:** Source code is saved using `%%writefile file.py`, then invoked with `!python file.py`. The table viewer must use `%run db_view.py` to display interactive Colab output in the notebook kernel. Run cells **in order**.

> SQLite is already included with Python as `sqlite3`; no SQLite pip install is needed. Colab runtime files are temporary unless saved to Drive.

## Part A1 — Database setup and exploration

The **books** table contains one row per book; the **reviews** table contains many reviews per book; and the **orders** table records purchases. These are linked by `book_id`.

The seed dataset has: **20 books and 25 reviews**. The setup is safe to rerun: it does not duplicate reviews or erase existing orders. Only the explicit `--reset` flag deletes the database.

In Part A, save each script with %%writefile and execute it with the following !python cell. Run the table viewer with %run db_view.py so its display appears in the notebook.

In [ ]:
%%writefile db_setup.py
"""Part A1 — Create and seed the bookstore's SQLite database.

Run: python db_setup.py
Fresh restart (erases existing orders and any edits): python db_setup.py --reset
"""

import argparse
import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).resolve().parent / "bookstore.db"

# dataset: 20 books and 25 reviews.
BOOKS = [(1, 'Dune', 'Frank Herbert', 'sci-fi', 14.99, 'Paperback', 1),
 (2, 'Project Hail Mary', 'Andy Weir', 'sci-fi', 18.99, 'Hardcover', 1),
 (3, 'Neuromancer', 'William Gibson', 'sci-fi', 10.99, 'Paperback', 0),
 (4, 'Foundation', 'Isaac Asimov', 'sci-fi', 12.49, 'Paperback', 0),
 (5, 'The Hobbit', 'J.R.R. Tolkien', 'fantasy', 13.99, 'Paperback', 1),
 (6, 'The Name of the Wind', 'Patrick Rothfuss', 'fantasy', 16.99, 'Paperback', 0),
 (7, 'The Way of Kings', 'Brandon Sanderson', 'fantasy', 24.99, 'Hardcover', 1),
 (8, 'The Silent Patient', 'Alex Michaelides', 'mystery', 11.99, 'Paperback', 1),
 (9, 'And Then There Were None', 'Agatha Christie', 'mystery', 9.99, 'Paperback', 0),
 (10, 'Gone Girl', 'Gillian Flynn', 'mystery', 13.49, 'Paperback', 0),
 (11, '1984', 'George Orwell', 'classics', 9.49, 'Paperback', 1),
 (12, 'Brave New World', 'Aldous Huxley', 'classics', 10.99, 'Paperback', 0),
 (13, 'To Kill a Mockingbird', 'Harper Lee', 'classics', 12.99, 'Paperback', 1),
 (14, 'Atomic Habits', 'James Clear', 'non-fiction', 16.99, 'Hardcover', 1),
 (15, 'Sapiens', 'Yuval Noah Harari', 'non-fiction', 15.49, 'Paperback', 1),
 (16, 'Clean Code', 'Robert C. Martin', 'tech', 34.99, 'Paperback', 0),
 (17,
  'Designing Data-Intensive Applications',
  'Martin Kleppmann',
  'tech',
  38.99,
  'Paperback',
  1),
 (18, 'Deep Learning', 'Ian Goodfellow', 'tech', 49.99, 'Hardcover', 0),
 (19, 'Meditations', 'Marcus Aurelius', 'philosophy', 8.99, 'Paperback', 1),
 (20, 'Beyond Good and Evil', 'Friedrich Nietzsche', 'philosophy', 9.99, 'Paperback', 0)]

REVIEWS = [(1, 5.0, 'Paul', 'An absolute sci-fi masterpiece of world-building.'),
 (1, 4.5, 'Jessica', 'Complex politics, unmatched depth.'),
 (1, 4.5, 'Chani', 'Classic sci-fi that still holds up.'),
 (2, 5.0, 'Rocky', 'Amazingly fun, hard sci-fi at its best!'),
 (2, 5.0, 'Stratt', "Couldn't put it down. 5 stars!"),
 (3, 4.0, 'Case', 'Gritty cyberpunk atmosphere, a bit dense.'),
 (3, 3.5, 'Molly', 'Influential, but pacing feels uneven.'),
 (5, 5.0, 'Bilbo', 'Charming adventure, perfect fantasy.'),
 (5, 4.5, 'Gandalf', 'Timeless story for all ages.'),
 (7, 5.0, 'Kaladin', 'Epic scale, incredible magic system.'),
 (7, 4.5, 'Shallan', 'Huge book, worth every single page.'),
 (8, 4.5, 'Theo', 'Great psychological plot twist at the end.'),
 (8, 3.5, 'Alicia', 'Pacing was good, ending was a bit rushed.'),
 (9, 5.0, 'Vera', 'The greatest mystery ever written.'),
 (9, 4.5, 'Lombard', 'Brilliant ending, totally unexpected.'),
 (11, 5.0, 'Winston', 'Chilling and deeply prophetic.'),
 (11, 4.5, 'Julia', 'Essential reading for everyone.'),
 (14, 5.0, 'Mark', 'Practical framework that actually changes habits.'),
 (14, 4.5, 'Sarah', 'Super actionable insights.'),
 (15, 4.5, 'Noah', 'Fascinating overview of human history.'),
 (15, 4.0, 'Elena', 'Engaging storytelling, broad generalizations.'),
 (17, 5.0, 'Dave', 'The definitive guide to distributed systems.'),
 (17, 5.0, 'Alice', 'Must-read for every backend engineer.'),
 (19, 5.0, 'StoicFan', 'Life-changing daily wisdom.'),
 (19, 4.5, 'Marcus', 'Remarkably modern advice from ancient Rome.')]


def setup_bookstore_db(reset=False):
    """Create tables and insert seed data without duplicating it on reruns."""
    if reset and DB_PATH.exists():
        DB_PATH.unlink()  # Explicit reset: also removes existing orders.

    conn = sqlite3.connect(DB_PATH)
    try:
        conn.execute("PRAGMA foreign_keys = ON")
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS books (
                id INTEGER PRIMARY KEY,
                title TEXT NOT NULL,
                author TEXT NOT NULL,
                genre TEXT NOT NULL,
                price REAL NOT NULL,
                format TEXT NOT NULL,
                is_bestseller INTEGER DEFAULT 0 CHECK (is_bestseller IN (0, 1))
            );
            CREATE TABLE IF NOT EXISTS reviews (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                book_id INTEGER NOT NULL,
                rating REAL NOT NULL CHECK (rating BETWEEN 0 AND 5),
                reviewer_name TEXT,
                review_text TEXT,
                FOREIGN KEY (book_id) REFERENCES books(id)
            );
            CREATE TABLE IF NOT EXISTS orders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                book_id INTEGER NOT NULL,
                book_title TEXT NOT NULL,
                price REAL NOT NULL,
                ordered_at TEXT NOT NULL DEFAULT (datetime('now')),
                FOREIGN KEY (book_id) REFERENCES books(id)
            );
        """)
        conn.executemany(
            """INSERT OR IGNORE INTO books
               (id, title, author, genre, price, format, is_bestseller)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            BOOKS,
        )
        # No duplicate reviews on rerun. A --reset provides a fresh dataset.
        if conn.execute("SELECT COUNT(*) FROM reviews").fetchone()[0] == 0:
            conn.executemany(
                """INSERT INTO reviews
                   (book_id, rating, reviewer_name, review_text)
                   VALUES (?, ?, ?, ?)""",
                REVIEWS,
            )
        conn.commit()
        for name in ("books", "reviews", "orders"):
            count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
            print(f"{name:>7}: {count} rows")
        print(f"Database ready: {DB_PATH}")
    finally:
        conn.close()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Initialize the sample bookstore")
    parser.add_argument(
        "--reset", action="store_true",
        help="Erase and recreate the database, including all orders",
    )
    args = parser.parse_args()
    setup_bookstore_db(reset=args.reset)

In [ ]:
!python db_setup.py

### Interactive data table

We save the viewer as a standalone Python file, then execute it **inside the notebook kernel** with `%run`. The result is an interactive Colab table (sortable and searchable). It also joins review data to compute average rating.

In [ ]:
%%writefile db_view.py
"""Part A1 — Interactive bookstore table. Use `%run db_view.py` in Colab."""

import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd
from IPython.display import display

DB_PATH = Path(__file__).resolve().parent / "bookstore.db"


def load_catalog():
    """Return one row per book with average review rating (where available)."""
    sql = """
        SELECT b.id, b.title, b.author, b.genre, b.price, b.format,
               b.is_bestseller,
               COUNT(r.id) AS review_count,
               ROUND(AVG(r.rating), 2) AS average_rating
        FROM books AS b
        LEFT JOIN reviews AS r ON r.book_id = b.id
        GROUP BY b.id
        ORDER BY b.id
    """
    with closing(sqlite3.connect(DB_PATH)) as conn:
        return pd.read_sql_query(sql, conn)


def show_catalog():
    df = load_catalog()
    print(f"Displaying {len(df)} books. Null ratings mean no reviews yet.")
    try:
        from google.colab import data_table
    except ImportError:
        display(df)  # Fallback for Jupyter outside Colab.
    else:
        display(data_table.DataTable(
            df, include_index=False, num_rows_per_page=10
        ))
    return df


if __name__ == "__main__":
    catalog_df = show_catalog()

In [ ]:
%run db_view.py

**Questions — A1 (write short answers).**

1. **What** is the relationship between `books` and `reviews`? Why not put all reviews directly in the `books` table?
2. **Why** does SQLite store `is_bestseller` as `0` or `1` rather than a Python `bool`?
3. **What** does an empty average rating mean? Is it the same thing as an average rating of zero?
4. **How** does the `LEFT JOIN` affect books with no reviews? What would change with an `INNER JOIN`?
5. **Why** are repeated setup runs designed not to duplicate reviews or delete orders?

**Experiment A1:** In `db_view.py`, edit `load_catalog()` so the table shows **only sci-fi books**, then restore the full catalog. Predict the number of rows before running.

**Expected baseline:** 20 books, 25 reviews, 0 orders on a new database. Running setup again leaves the counts unchanged.

## Part A2 — Plain Python API (not LLM tools yet)

We use the following API function names:

- `search_books(...)` — return matching books with optional filters.
- `get_book_rating(book_id)` — return average rating and number of reviews.
- `order_book(book_id)` — insert a purchase in the orders table.

API behavior: SQL values use parameters, Boolean strings are normalized, and missing books are distinguished from books without reviews. Running books_api.py performs read-only demonstrations. The order_book function inserts a simulated order in the teaching database; it is not exposed to the agent anywhere in this notebook. A production purchase workflow would also need authorization and user confirmation

In [ ]:
%%writefile books_api.py
"""Part A2 — Standard Python functions, BEFORE exposing them to an LLM.

Running this file performs READ-ONLY demonstrations; it never places an order.
"""

import sqlite3
from contextlib import closing
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

DB_PATH = Path(__file__).resolve().parent / "bookstore.db"


def _connect():
    if not DB_PATH.exists():
        raise FileNotFoundError("Run `python db_setup.py` before using books_api.py")
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def _normalize_bestseller(value):
    """Accept strict Boolean values and common tool-call string encodings."""
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in ("true", "1", "yes"):
            return True
        if normalized in ("false", "0", "no"):
            return False
    if type(value) is int and value in (0, 1):
        return bool(value)
    raise ValueError("is_bestseller must be true or false (or 1/0)")


def _book_id(value):
    if isinstance(value, bool):
        raise ValueError("book_id must be a positive integer")
    try:
        number = int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError("book_id must be a positive integer") from exc
    if number <= 0 or str(value).strip() not in (str(number),):
        raise ValueError("book_id must be a positive integer")
    return number


def search_books(
    query: Optional[str] = None,
    genre: Optional[str] = None,
    max_price: Optional[float] = None,
    is_bestseller: Optional[Union[bool, str, int]] = None,
    limit: int = 10,
) -> List[Dict[str, Any]]:
    """Search the catalog; each filter is optional (uses SQL parameters)."""
    limit = int(limit)
    if not 1 <= limit <= 100:
        raise ValueError("limit must be between 1 and 100")
    if max_price is not None:
        max_price = float(max_price)
        if max_price < 0:
            raise ValueError("max_price cannot be negative")
    bestseller = _normalize_bestseller(is_bestseller)

    sql = """SELECT id, title, author, genre, price, format, is_bestseller
             FROM books WHERE 1 = 1"""
    params = []
    if query:
        sql += " AND (title LIKE ? OR author LIKE ?)"
        wildcard = f"%{query.strip()}%"
        params.extend((wildcard, wildcard))
    if genre:
        sql += " AND genre = ?"
        params.append(genre.strip().lower())
    if max_price is not None:
        sql += " AND price <= ?"
        params.append(max_price)
    if bestseller is not None:
        sql += " AND is_bestseller = ?"
        params.append(int(bestseller))
    sql += " ORDER BY id LIMIT ?"
    params.append(limit)

    with closing(_connect()) as conn:
        rows = conn.execute(sql, params).fetchall()
    return [
        {**dict(row), "is_bestseller": bool(row["is_bestseller"])}
        for row in rows
    ]


def get_book_rating(book_id: int) -> Dict[str, Any]:
    """Rating and review count; distinguish missing books from unrated books."""
    bid = _book_id(book_id)
    with closing(_connect()) as conn:
        row = conn.execute(
            """SELECT b.id, AVG(r.rating) AS avg_rating,
                      COUNT(r.id) AS review_count
               FROM books AS b
               LEFT JOIN reviews AS r ON r.book_id = b.id
               WHERE b.id = ?
               GROUP BY b.id""", (bid,),
        ).fetchone()
    if row is None:
        return {"book_id": bid, "found": False,
                "average_rating": None, "review_count": 0}
    return {
        "book_id": bid,
        "found": True,
        "average_rating": (
            round(row["avg_rating"], 2) if row["avg_rating"] is not None
            else None
        ),
        "review_count": row["review_count"],
    }


def order_book(book_id: int) -> Dict[str, Any]:
    """Place a REAL order in SQLite; future agents must seek confirmation first."""
    bid = _book_id(book_id)
    with closing(_connect()) as conn:
        row = conn.execute(
            "SELECT title, price FROM books WHERE id = ?", (bid,)
        ).fetchone()
        if row is None:
            return {"status": "error",
                    "message": f"Book with ID {bid} not found in catalog."}
        cursor = conn.execute(
            "INSERT INTO orders (book_id, book_title, price) VALUES (?, ?, ?)",
            (bid, row["title"], row["price"]),
        )
        conn.commit()
        new_order_id = cursor.lastrowid
    return {
        "status": "success",
        "order_id": new_order_id,
        "book_title": row["title"],
        "price": row["price"],
        "message": (f"Order #{new_order_id} confirmed! "
                    f"'{row['title']}' ordered for ${row['price']:.2f}."),
    }


if __name__ == "__main__":
    print("Search: sci-fi bestsellers for <= $16")
    print(search_books(genre="sci-fi", max_price=16, is_bestseller=True))
    print("\nDune rating:")
    print(get_book_rating(1))
    print("\nBoolean-string check: sci-fi NON-bestsellers")
    print([book["title"] for book in search_books(
        genre="sci-fi", is_bestseller="False")])
    print("\nMissing book rating:")
    print(get_book_rating(999))
    print("\nNo purchase was made by these demonstrations.")

In [ ]:
!python books_api.py

### Verify the API

The following checks test the API but **do not place an order**. Assertions make expected behavior explicit.

In [ ]:
%%writefile part_a_checks.py
"""Part A — Safe checks. Running this script does not create orders."""
import sqlite3
from contextlib import closing
from books_api import DB_PATH, search_books, get_book_rating, order_book

results = search_books(genre="sci-fi", max_price=16, is_bestseller=True)
assert [row["title"] for row in results] == ["Dune"]
assert all(isinstance(row["is_bestseller"], bool) for row in results)
non_best = search_books(genre="sci-fi", is_bestseller="False")
assert [row["title"] for row in non_best] == ["Neuromancer", "Foundation"]
assert get_book_rating(1)["average_rating"] == 4.67
assert get_book_rating(1)["review_count"] == 3
assert get_book_rating(999)["found"] is False
assert get_book_rating(4)["found"] is True
assert get_book_rating(4)["average_rating"] is None
assert order_book(999)["status"] == "error"  # No write on invalid ID.
with closing(sqlite3.connect(DB_PATH)) as conn:
    count = conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
print(f"All checks passed. Orders currently in DB: {count}")

In [ ]:
!python part_a_checks.py

**Questions — A2 (write short answers).**

1. **What** does `search_books(genre="sci-fi", max_price=16, is_bestseller=True)` return? Why?
2. **Why** use SQL placeholders (`?`) and separate `params` instead of embedding user input directly?
3. **Why** is `if is_bestseller:` unreliable for the string `'False'`? How does `_normalize_bestseller` fix it?
4. **How** do `search_books` and `order_book` differ in the external state they affect?
5. **What** is the difference between `get_book_rating(999)` and `get_book_rating(4)`?
6. **Why** should an agent ask for confirmation before placing an order?

**Experiment A2:** Edit the demonstration search in `books_api.py` to find fantasy books under $15. Predict the results first. Next, evaluate `search_books(is_bestseller="False")` and explain what happened.

**Checkpoint:** These are ordinary Python functions. They become *LLM tools* only when we register them with structured schemas and let a model request their execution in Parts C–E.

---------